In [2]:
import tensorflow as tf
from kapre import STFT, Magnitude, ApplyFilterbank, MagnitudeToDecibel, STFTTflite, MagnitudeTflite
import numpy as np
from pathlib import Path

import sys
sys.path.append("../")
from genetic_algorithm.utils.convert_to_tflite import convert_to_tflite
#from get_ops import get_ops

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

2024-03-21 13:51:42.518151: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-03-21 13:51:42.555385: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-03-21 13:51:42.555569: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [4]:
PATH = Path("../tools/test_models_pool/")

if not PATH.exists():
    PATH.mkdir()

In [3]:
def save_as_tflite(model, save_path, quantize, input_shape, print_ops=False):
    if quantize:
        tflite_model = convert_to_tflite(model, np.random.uniform(size=input_shape))
        save_path = str(save_path).replace(".tflite", "_quantized.tflite")
    else:
        tflite_model = convert_to_tflite(model)
        save_path = str(save_path).replace(".tflite", "_non_quantized.tflite")
        
    with open(save_path, 'wb') as f:
        f.write(tflite_model)

## Simple CNN model (input: 28x28x1)

In [6]:
name = "simple_cnn_28x28"
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(10, activation='softmax')
])

#model.summary()
model.save(PATH / (name + ".keras"))

save_as_tflite(model, PATH / (name + ".tflite"), quantize=True, input_shape=(1, 28, 28, 1), print_ops=False)
save_as_tflite(model, PATH / (name + ".tflite"), quantize=False, input_shape=(1, 28, 28, 1), print_ops=False)

INFO:tensorflow:Assets written to: /tmp/tmpym7t6xxg/assets


INFO:tensorflow:Assets written to: /tmp/tmpym7t6xxg/assets
/data/du92wufe/Documents/EvoNAS/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:766: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "
2023-11-20 19:03:02.685007: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2023-11-20 19:03:02.685020: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2023-11-20 19:03:02.685402: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /tmp/tmpym7t6xxg
2023-11-20 19:03:02.685994: I tensorflow/cc/saved_model/reader.cc:81] Reading meta graph with tags { serve }
2023-11-20 19:03:02.686005: I tensorflow/cc/saved_model/reader.cc:122] Reading SavedModel debug info (if present) from: /tmp/tmpym7t6xxg
2023-11-20 19:03:02.688556: I tensorflow/compil

INFO:tensorflow:Assets written to: /tmp/tmpn6bidypq/assets


INFO:tensorflow:Assets written to: /tmp/tmpn6bidypq/assets
2023-11-20 19:03:03.115150: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2023-11-20 19:03:03.115166: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2023-11-20 19:03:03.115270: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /tmp/tmpn6bidypq
2023-11-20 19:03:03.115876: I tensorflow/cc/saved_model/reader.cc:81] Reading meta graph with tags { serve }
2023-11-20 19:03:03.115887: I tensorflow/cc/saved_model/reader.cc:122] Reading SavedModel debug info (if present) from: /tmp/tmpn6bidypq
2023-11-20 19:03:03.117646: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2023-11-20 19:03:03.126397: I tensorflow/cc/saved_model/loader.cc:212] Running initialization op on SavedModel bundle at path: /tmp/tmpn6bidypq
2023-11-20 19:03:03.130827: I tensorflow/cc/saved_model/loader.cc:301] SavedModel

## Simple CNN model (input: 256x256x1)

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.summary()

model.save(PATH / "simple_cnn_256x256.keras")

## STFT CNN model (input: 2048x1)

In [125]:
input_shape = (8000, 1)
model = tf.keras.models.Sequential()
model.add(tf.keras.Input(shape=input_shape))

# get number of layers in model
n_layers = len(model.layers)
n_layers

0

In [7]:
NAME = "spoken_languages_1d_and_stft.h5"
model = tf.keras.models.Sequential()

input_shape = (8000, 1)
model.add(tf.keras.Input(shape=input_shape))

# add some 1D convolutions
model.add(tf.keras.layers.DepthwiseConv1D(1, 1, padding='same'))

#model.add(tf.keras.layers.GlobalAveragePooling1D())

model.add(STFTTflite(n_fft=64, hop_length=396,
              input_data_format='channels_last',
              output_data_format='channels_last',
              name='stft'))

model.add(MagnitudeTflite(name='magnitude'))
model.add(tf.keras.layers.DepthwiseConv2D(1, 1, padding='same'))
model.add(tf.keras.layers.GlobalAveragePooling2D())
model.add(tf.keras.layers.Dense(4, activation='softmax'))

model.summary()
model.save(PATH / NAME, save_format='keras')

save_as_tflite(model, PATH / NAME.replace("h5", "tflite"), quantize=True, input_shape=(1,) + input_shape, print_ops=False)
#save_as_tflite(model, PATH / NAME.replace("keras", "tflite"), quantize=False, input_shape=(1, 6000, 1), print_ops=False)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 depthwise_conv1d_1 (Depthwi  (None, 8000, 1)          2         
 seConv1D)                                                       
                                                                 
 stft (STFTTflite)           (1, 21, 33, 1, 2)         0         
                                                                 
 magnitude (MagnitudeTflite)  (1, 21, 33, 1)           0         
                                                                 
 depthwise_conv2d_1 (Depthwi  (1, 21, 33, 1)           2         
 seConv2D)                                                       
                                                                 
 global_average_pooling2d_1   (1, 1)                   0         
 (GlobalAveragePooling2D)                                        
                                                      

INFO:tensorflow:Assets written to: /tmp/tmpg9rsy_nn/assets


INFO:tensorflow:Assets written to: /tmp/tmpg9rsy_nn/assets
/data/du92wufe/Documents/EvoNAS/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:766: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "
2024-01-07 16:21:11.038607: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2024-01-07 16:21:11.038627: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2024-01-07 16:21:11.038764: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /tmp/tmpg9rsy_nn
2024-01-07 16:21:11.040198: I tensorflow/cc/saved_model/reader.cc:81] Reading meta graph with tags { serve }
2024-01-07 16:21:11.040212: I tensorflow/cc/saved_model/reader.cc:122] Reading SavedModel debug info (if present) from: /tmp/tmpg9rsy_nn
2024-01-07 16:21:11.045548: I tensorflow/cc/sav


## More networks: TODO: clean-up the code that follows

In [3]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=6_000),
    STFT(frame_length=512, frame_step=128, n_freqs=40, out_module=False)
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

2023-10-30 13:43:19.170897: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-10-30 13:43:19.229241: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-10-30 13:43:19.229376: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-10-30 13:43:19.230224: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the approp

In [4]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 stft (STFT)                 (None, 42, 40, 2)         552       
                                                                 
Total params: 552
Trainable params: 0
Non-trainable params: 552
_________________________________________________________________


In [23]:
model = tf.keras.Sequential()

#model.add(tf.keras.Input(shape=6000))
model.add(tf.keras.Input(shape=(6000, 1)))

# model.add(STFTTflite(n_fft=400, hop_length=160,
#               input_data_format='channels_last',
#               output_data_format='channels_last',
#               input_shape=(6000, 1),
#               name='stft'))

#model.add(TF_STFT_Layer(name='stft'))
model.add(STFTTflite(n_fft=512, hop_length=128, input_data_format='channels_last',
              output_data_format='channels_last',
              input_shape=(6000, 1),
              name='stft'))

model.add(MagnitudeTflite(name='magnitude'))


model.add(tf.keras.layers.Conv2D(32, 3))
model.add(tf.keras.layers.ReLU())
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPooling2D())

model.add(tf.keras.layers.Conv2D(16, 3))
model.add(tf.keras.layers.ReLU())
model.add(tf.keras.layers.BatchNormalization())

model.add(tf.keras.layers.GlobalAveragePooling2D())
model.add(tf.keras.layers.Dense(12, activation='relu'))

In [24]:
model.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 stft (STFTTflite)           (1, 43, 257, 1, 2)        0         
                                                                 
 magnitude (MagnitudeTflite)  (1, 43, 257, 1)          0         
                                                                 
 conv2d_8 (Conv2D)           (1, 41, 255, 32)          320       
                                                                 
 re_lu_8 (ReLU)              (1, 41, 255, 32)          0         
                                                                 
 batch_normalization_8 (Batc  (1, 41, 255, 32)         128       
 hNormalization)                                                 
                                                                 
 max_pooling2d_4 (MaxPooling  (1, 20, 127, 32)         0         
 2D)                                                  

In [25]:
#tflite_model = convert_to_tflite(model, np.random.uniform(size=(1, 6_000)))
tflite_model = convert_to_tflite(model)

INFO:tensorflow:Assets written to: /tmp/tmpujvnc6qg/assets


INFO:tensorflow:Assets written to: /tmp/tmpujvnc6qg/assets


Estimated count of arithmetic ops: 50.872 M  ops, equivalently 25.436 M  MACs


2024-03-21 14:04:16.033859: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2024-03-21 14:04:16.033877: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2024-03-21 14:04:16.033966: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /tmp/tmpujvnc6qg
2024-03-21 14:04:16.035145: I tensorflow/cc/saved_model/reader.cc:81] Reading meta graph with tags { serve }
2024-03-21 14:04:16.035157: I tensorflow/cc/saved_model/reader.cc:122] Reading SavedModel debug info (if present) from: /tmp/tmpujvnc6qg
2024-03-21 14:04:16.038484: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2024-03-21 14:04:16.055187: I tensorflow/cc/saved_model/loader.cc:212] Running initialization op on SavedModel bundle at path: /tmp/tmpujvnc6qg
2024-03-21 14:04:16.063440: I tensorflow/cc/saved_model/loader.cc:301] SavedModel load for tags { serve }; Status: success: OK. Took 29475 m

In [26]:
with open('../tools/test_models_pool/test_model_custom_stft.tflite', 'wb') as f:
    f.write(tflite_model)

In [44]:
get_ops('test_model_custom_stft.tflite')

['EXPAND_DIMS (0)',
 'MUL (1)',
 'TRANSPOSE (2)',
 'SHAPE (3)',
 'GATHER (4)',
 'REDUCE_PROD (5)',
 'CONCATENATION (6)',
 'GATHER (4)',
 'REDUCE_PROD (5)',
 'PACK (7)',
 'RESHAPE (8)',
 'FULLY_CONNECTED (9)',
 'RESHAPE (8)',
 'FULLY_CONNECTED (9)',
 'RESHAPE (8)',
 'PACK (7)',
 'MUL (1)',
 'MUL (1)',
 'SUM (10)',
 'DEQUANTIZE (11)',
 'SQRT (12)',
 'QUANTIZE (13)',
 'SQUEEZE (14)',
 'EXPAND_DIMS (0)',
 'CONV_2D (15)',
 'MUL (1)',
 'ADD (16)',
 'MAX_POOL_2D (17)',
 'CONV_2D (15)',
 'MUL (1)',
 'ADD (16)',
 'MEAN (18)',
 'FULLY_CONNECTED (9)']